[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/soccer-vision/blob/master/examples/acceptance_pitch.ipynb)

# Pitch-v1 acceptance gate (Phase 3.5b)
Runs the pipeline on the held-out multi-field set (bake-off clip + clips from
>=2 other games/fields, >=1 entirely unseen in training).
Gates (aggregate): anchor coverage >= 40%, combined coverage >= 65%,
held-out reproj error <= 0.05 on every clip. Reports per-clip + per-field,
split unseen-field vs unseen-time.

In [ ]:
!pip install -q "soccer-vision[roboflow] @ git+https://github.com/PatrickJReed/soccer-vision.git#subdirectory=packages/soccer-vision"
from pathlib import Path

WEIGHTS = Path("/content/pitch_v1.pt")
BAKEOFF = Path("/content/bakeoff_clip.mp4")
HELDOUT = Path("/content/heldout_gameC.mp4")  # unseen field (Chula Vista)
# stage weights + clip from Drive soccer-vision/ if not already uploaded
if not WEIGHTS.exists() or not BAKEOFF.exists() or not HELDOUT.exists():
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/soccer-vision")
    if not WEIGHTS.exists():
        # last.pt (e42) scored higher on POSE mAP than best.pt (e22) this run
        for cand in ("last.pt", "best.pt", "pitch_v1.pt"):
            if (DRIVE / cand).exists():
                !cp "{DRIVE/cand}" /content/pitch_v1.pt
                print("using weights:", cand); break
    if not BAKEOFF.exists() and (DRIVE / "clip.mp4").exists():
        !cp "{DRIVE/'clip.mp4'}" /content/bakeoff_clip.mp4
    if not HELDOUT.exists() and (DRIVE / "heldout_gameC.mp4").exists():
        !cp "{DRIVE/'heldout_gameC.mp4'}" /content/heldout_gameC.mp4
assert WEIGHTS.exists(), "upload pitch weights or place last.pt in Drive soccer-vision/"
assert BAKEOFF.exists(), "upload the clip or place clip.mp4 in Drive soccer-vision/"
MAX_GAP = 45

# INTERIM (today): bake-off only — a SEEN field, so coverage is FLATTERED, but it
# shows whether the model yields usable anchors through the real pipeline at all.
# Baseline to beat: broadcast model gave ~16% anchor coverage on this clip.
# FULL gate: add >=1 game the model NEVER trained on (unseen_field) — the real test.
CLIPS = [
    (Path("/content/bakeoff_clip.mp4"), "bakeoff", "seen_field_interim"),
    (Path("/content/heldout_gameC.mp4"), "chula_vista", "unseen_field"),
]


In [ ]:
import cv2
from soccer_vision.tracking.roboflow import RoboflowBackend
from soccer_vision.pitch.landmarks import build_frame_homographies
from soccer_vision.pitch.propagation import (
    compute_interframe_homographies, propagate_homographies,
)

backend = RoboflowBackend(detect_pitch=True, pitch_weights_path=WEIGHTS)
rows = []
for path, field, split in CLIPS:
    traj, kps = backend.process_with_pitch(path)
    cap = cv2.VideoCapture(str(path))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    def read_frame(i, _cap=cap):
        _cap.set(cv2.CAP_PROP_POS_FRAMES, i); ok, f = _cap.read(); return f if ok else None
    anchors = build_frame_homographies(kps)
    keys = sorted(anchors)
    needed = {i for a, b in zip(keys, keys[1:]) if b - a - 1 <= MAX_GAP for i in range(a, b)}
    interframe = compute_interframe_homographies(read_frame, needed, traj)
    homs = propagate_homographies(anchors, interframe, max_gap=MAX_GAP)
    cap.release()
    anchor_cov = len(anchors) / n
    combined_cov = len(homs) / n
    rows.append(dict(field=field, split=split, frames=n,
                     anchor_cov=anchor_cov, combined_cov=combined_cov))
    print(f"{field:8s} [{split}] anchor={anchor_cov:.1%} combined={combined_cov:.1%}")

In [ ]:
import pandas as pd
df = pd.DataFrame(rows)
agg_anchor = df["anchor_cov"].mean()
agg_combined = df["combined_cov"].mean()
print(f"AGG anchor={agg_anchor:.1%} (gate >=40%): {'PASS' if agg_anchor>=0.40 else 'FAIL'}")
print(f"AGG combined={agg_combined:.1%} (gate >=65%): {'PASS' if agg_combined>=0.65 else 'FAIL'}")
print("baseline: broadcast pitch model gave ~16% anchor coverage on the bake-off clip\n")
if (df['split'] == 'unseen_field').any():
    print("By split:")
    print(df.groupby('split')[['anchor_cov','combined_cov']].mean())
else:
    print("INTERIM run (seen field only) — coverage is flattered; add an unseen "
          "game for the real gate.")
print("\nWorst field (anchor):", df.loc[df['anchor_cov'].idxmin(), 'field'])


Reprojection error and keypoint accuracy reuse the 3.5a held-out probe
(examples/colab_homography_propagation.ipynb) and a small labeled slice per
clip. Report median per-keypoint pixel error + per-landmark detection rate,
split unseen-field vs unseen-time. Reproj error must stay <= 0.05 on every clip.

## Accuracy check — reprojected pitch on the UNSEEN field
Coverage says a homography *fits*; this says it's *accurate*. Runs the pitch
model on sampled Chula Vista frames and overlays detected keypoints (green) +
the pitch reprojected through the inverse homography (orange). 3.5b is truly
done if the orange lines trace the painted markings.


In [ ]:
import random

import cv2
import numpy as np
from ultralytics import YOLO
from soccer_vision.viz.pitch_overlay import draw_reprojected_pitch

CLIP = Path("/content/heldout_gameC.mp4")  # the unseen field
model = YOLO(str(WEIGHTS))
cap = cv2.VideoCapture(str(CLIP))
n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
random.seed(0)
sample = sorted(random.sample(range(n), 12))

cells, n_fit = [], 0
for fi in sample:
    cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
    ok, frame = cap.read()
    if not ok:
        continue
    res = model.predict(frame, imgsz=1280, verbose=False)[0]
    kp = res.keypoints
    if kp is None or kp.xy is None or len(kp.xy) == 0:
        cells.append(frame); continue
    xy = kp.xy[0].cpu().numpy()            # (21, 2) px
    cf = kp.conf[0].cpu().numpy()          # (21,)
    keep = cf >= 0.5
    idx = np.nonzero(keep)[0]
    annotated, fit_ok = draw_reprojected_pitch(frame, xy[keep], idx)
    n_fit += int(fit_ok)
    cells.append(annotated)
cap.release()
print(f"{n_fit}/{len(cells)} sampled frames fit a homography")

# tile into a 4-col contact sheet (downscale-survivable markers were drawn at full res)
cw = 640
tiles = [cv2.resize(c, (cw, int(c.shape[0] * cw / c.shape[1]))) for c in cells]
rows_img = []
for i in range(0, len(tiles), 4):
    row = tiles[i:i + 4]
    row += [np.zeros_like(tiles[0])] * (4 - len(row))
    rows_img.append(np.hstack(row))
sheet = np.vstack(rows_img)
cv2.imwrite("/content/unseen_overlay.jpg", sheet)
from IPython.display import Image, display
display(Image("/content/unseen_overlay.jpg"))
